# Build a risk-analyst LangChain agent in 25 lines

**Problem:** LLMs are unreliable at *computing* finance math. Ask one to price a Black-Scholes call and the answer drifts run to run. Ask for the higher-order Greeks and they're frequently wrong.

**Fix:** keep the LLM for reasoning, hand the math to a deterministic API.

This notebook shows a LangChain agent that answers risk and portfolio questions using [QuantOracle](https://api.quantoracle.dev) — 63 deterministic quant calculators + 10 composite workflows, all citation-verified against Hull / Wilmott / Bailey & Lopez de Prado.

- 1,000 free calls per IP per day, no signup, no API key
- Paid composites via [x402](https://x402.org) USDC micropayments on Base or Solana
- Cataloged in [CDP Bazaar](https://api.cdp.coinbase.com/platform/v2/x402/discovery/resources) for agent discovery

## 1. Install

In [ ]:
%pip install --quiet langchain-quantoracle langchain-openai langchain

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = "sk-..."  # your OpenAI key

## 2. Load just the risk + portfolio + stats tools

QuantOracle exposes 73 endpoints. Loading all of them into an agent prompt confuses smaller models. Filter by category to keep the tool list focused.

In [ ]:
from langchain_quantoracle import QuantOracleToolkit

tools = QuantOracleToolkit(categories=["risk", "portfolio", "stats"]).get_tools()
print(f"Loaded {len(tools)} tools")
for t in tools[:5]:
    print(f"  • {t.name}")
print("  ...")

## 3. Build the agent

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a quantitative analyst. For ANY financial calculation — Sharpe, "
     "VaR, drawdown, Kelly, position sizing, regression — you MUST call a "
     "QuantOracle tool. Never compute math in your head. After tools return, "
     "explain the result in plain English and recommend an action."),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

agent = create_tool_calling_agent(llm, tools, prompt)
executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

## 4. Ask a risk question with real returns

These are 20 days of made-up daily returns. The agent should pick `risk_portfolio` (a free tool that bundles Sharpe, Sortino, Calmar, VaR, CVaR, drawdown, win rate, fat-tail check) and contextualize the result.

In [ ]:
returns = [
    0.012, -0.008, 0.015, -0.022, 0.018, 0.005, -0.011, 0.014, -0.003, 0.009,
    -0.018, 0.020, -0.007, 0.012, 0.003, -0.013, 0.016, -0.005, 0.011, -0.009,
]

result = executor.invoke({
    "input": f"Here are my last 20 daily returns: {returns}. "
             "Am I taking too much risk for the return I'm getting? "
             "Give me Sharpe, max drawdown, and one sentence of advice."
})

print("\n--- AGENT ANSWER ---\n")
print(result["output"])

Behind the scenes that one call returned (real API output, deterministic):

```json
{
  "returns": {"annualized": 0.4914, "vol": 0.2052, "win_rate": 0.55, "profit_factor": 1.4062},
  "risk":    {"sharpe": 2.151, "sortino": 3.3664, "calmar": 22.3364, "max_drawdown": -0.022,
              "var_95": -0.022, "cvar_95": -0.022},
  "distribution": {"skewness": -0.2911, "excess_kurtosis": -1.2361, "fat_tails": false},
  "n": 20, "ms": 18.5
}
```

Same input → same output, every time. The LLM doesn't *do* the math — it interprets the result.

## 5. Position sizing with Kelly criterion

In [ ]:
result = executor.invoke({
    "input": "I have a strategy with a 55% win rate. Average winner: $1.50. "
             "Average loser: $1.00. What fraction of my account should each "
             "trade risk? Give me full Kelly, half-Kelly, and quarter-Kelly."
})

print("\n--- AGENT ANSWER ---\n")
print(result["output"])

## 6. Composite workflows: when one tool replaces ten

QuantOracle has 10 **composite** endpoints that bundle 5–15 calculator calls into a single round trip with a purpose-built output. They're paid-only via x402 (USDC on Base or Solana).

Examples:
- `backtest_strategy` — full SMA / RSI / momentum backtest with equity curve + buy-hold benchmark ($0.10)
- `portfolio_rebalance-plan` — generate the trade list to reach target weights with cost estimate ($0.05)
- `options_strategy-optimizer` — rank top option strategies given your outlook + vol view ($0.08)
- `hedging_recommend` — compare protective puts, collars, inverse hedges ($0.04)
- `risk_full-analysis` — full tearsheet from a returns array ($0.04)

If you're using an x402-capable HTTP client (e.g. [AgentCash](https://agentcash.dev), Coinbase AgentKit), payment is automatic — your agent gets a 402, signs an EIP-3009 USDC transfer, and retries. Otherwise the toolkit raises an `HTTPError` and you can layer your own payment flow.

```python
# Example — same pattern, just include the composite categories
tools = QuantOracleToolkit(
    categories=["risk", "backtest", "portfolio", "hedging"]
).get_tools()
```

## What this pattern buys you

| | LLM-only | LLM + QuantOracle |
|---|---|---|
| Reproducibility | drifts run-to-run | deterministic |
| Higher-order Greeks (vanna, charm, speed) | frequently wrong | exact |
| Iterative numerics (IV solver, GARCH, optimizer) | degrades | converges |
| Cost per call | LLM tokens (~$0.02–0.10) | $0 free tier, then $0.002–$0.10 |
| Audit trail | none | every call hashable + replayable |

When an agent makes 50 tool calls during a backtest, every calculation has to be right. The pattern — **LLM for reasoning, deterministic API for compute** — is what actually works in production.

## Links

- API + docs: <https://api.quantoracle.dev/docs>
- Tool catalog: <https://api.quantoracle.dev/tools>
- GitHub: <https://github.com/QuantOracledev/quantoracle>
- PyPI: [`langchain-quantoracle`](https://pypi.org/project/langchain-quantoracle/)
- MCP server: `npx quantoracle-mcp` ([npm](https://www.npmjs.com/package/quantoracle-mcp))
- CDP Bazaar listings: <https://api.cdp.coinbase.com/platform/v2/x402/discovery/resources> (filter `resource` for `quantoracle.dev`)
- x402 discovery feed: <https://api.quantoracle.dev/.well-known/x402>